# Let's work on finding the correlation between measles coverage and incidence

### Let's use
- rada_notebooks/nis_measles_vacc_coverage.csv (Primary coverage)
- raw/Vaccination_Coverage_and_Exemptions_among_Kindergartners_20260827.csv (Secondary coverage)
- app/data/tycho_measles_control.csv, Tycho Measles (Primary incidents)

### Start with measles incidence/cases - we are now using Tycho's API for the most recent data

In [20]:
import pandas as pd

tycho_incidents_df = pd.read_csv('../app/data/tycho_measles_control.csv')
tycho_incidents_df.head()

,week,state,cases,year
0,1,NY,376.0,1931
1,1,OR,67.0,1931
2,1,CO,41.0,1931
3,1,AZ,50.0,1931
4,1,MO,1160.0,1931


### Let's start with graphing average measles incidience: year x cases

In [21]:
year_cases_nis_df = tycho_incidents_df.groupby(by='year')['cases'].mean().reset_index()
year_cases_nis_df.head()

,year,cases
0,1931,184.207077
1,1932,161.751344
2,1933,156.138298
3,1934,298.681465
4,1935,295.674304


In [22]:
year_cases_nis_df['year'].min(), year_cases_nis_df['year'].max()

(np.int64(1931), np.int64(1992))

In [23]:
import plotly.express as px

year_cases_nis_df_fig = px.line(
    year_cases_nis_df,
    x="year",
    y="cases",
    markers=True,
    title="Avg. Cases per Year"
)

year_cases_nis_df_fig.update_layout(
    xaxis_title="Year",
    yaxis_title="Avg. Measles Cases",
)

year_cases_nis_df_fig.show()

### Our primary coverage will be NIS Child (2015 - 2024)

In [24]:
nis_coverage_df = pd.read_csv('../app/data/nis_measles_vacc_coverage.csv')
nis_coverage_df.head()

,state,year,n,n_vaccinated,coverage_pct
0,AK,2015,295.0,262.0,89.70
1,AK,2016,288.0,253.0,85.83
2,AK,2017,251.0,221.0,89.56
3,AK,2018,228.0,191.0,85.11
4,AK,2019,240.0,204.0,85.79


In [25]:
nis_coverage_df['year'].min(), nis_coverage_df['year'].max()

(np.int64(2015), np.int64(2024))

## Recall what these columns mean:

| Column         | Level          | What it means                              | Why you need it                                                                     |
| -------------- | -------------- | ------------------------------------------ | ----------------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier                    | Identifies the individual child/record                                              |
| **`SEQNUMHH`** | Household      | Household identifier                       | Identifies which household the child belongs to; used in survey design              |
| **`STRATUM`**  | Sampling group | Survey sampling stratum                    | Identifies the sampling group; needed for correct SEs/CIs                           |
| **`PROVWT`**   | Child          | Provider-phase survey weight               | Determines how much the child contributes to population estimates                   |
| **`P_UTDMCV`** | Child          | Measles vaccination status                 | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not          |
| **`P_NUMMMR`** | Child          | Number of measles-containing vaccine doses | Shows how many provider-reported measles-containing vaccinations the child received |
| **`STATE`**    | Geography      | State code                                 | Lets you calculate vaccination coverage by state                                    |
| **`YEAR`**     | Time           | Survey year                                | Lets you calculate and compare coverage over time                                   |

In [26]:
assert False

AssertionError: 

In [1]:
import pandas as pd

schoolvaxview_df = pd.read_csv('../raw/school_vax_view/schoolvaxview.csv')
schoolvaxview_df.head()

,vaccine,dose,geography_type,geography,year_season,coverage_estimate,population_sample_size,percent_surveyed,foot_notes,number_of_exemptions,survey_type
0,MMR,NaN,States,Kansas,2022-23,91.6,35543.0,30.8,†. ‡. §,NaN,Stratified 1-stage cluster sample
1,MMR,NaN,States,Kentucky,2022-23,90.1,54742.0,96.9,>=. ‡. §,NaN,Census
2,MMR,NaN,States,Louisiana,2022-23,92.2,54314.0,100.0,*,NaN,Census
3,MMR,NaN,States,Maine,2022-23,96.8,12403.0,93.9,NaN,NaN,Census
4,MMR,NaN,States,Maryland,2022-23,96.7,59684.0,100.0,*. ‡,NaN,Census


In [2]:
schoolvaxview_df["year"] = (
    schoolvaxview_df["year_season"]
    .str.split("-")
    .str[0]
    .astype(int)
    .add(1)
)

In [3]:
import us


schoolvaxview_df["state"] = schoolvaxview_df["geography"].map(
    lambda x: us.states.lookup(x).abbr if us.states.lookup(x) else None
)


In [5]:
core_columns = [
    'vaccine',
    'dose',
    'geography_type',
    'coverage_estimate',
    'year',
    'state',
]

schoolvaxview_df = schoolvaxview_df[core_columns]
schoolvaxview_df.head()

,vaccine,dose,geography_type,coverage_estimate,year,state
0,MMR,NaN,States,91.6,2023,KS
1,MMR,NaN,States,90.1,2023,KY
2,MMR,NaN,States,92.2,2023,LA
3,MMR,NaN,States,96.8,2023,ME
4,MMR,NaN,States,96.7,2023,MD


In [8]:
measles_schoolvaxview_df = schoolvaxview_df[(schoolvaxview_df['vaccine'] == 'MMR')
                                            & (schoolvaxview_df['geography_type'] == 'States')]
len(measles_schoolvaxview_df)

868

In [10]:
measles_schoolvaxview_df.head()

,vaccine,dose,geography_type,coverage_estimate,year,state
0,MMR,NaN,States,91.6,2023,KS
1,MMR,NaN,States,90.1,2023,KY
2,MMR,NaN,States,92.2,2023,LA
3,MMR,NaN,States,96.8,2023,ME
4,MMR,NaN,States,96.7,2023,MD


In [11]:
measles_schoolvaxview_df['year'].min(), measles_schoolvaxview_df['year'].max(), 

(np.int64(2010), np.int64(2026))